In [1]:
import sqlite3
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#loading the cleaned data 
df = pd.read_csv('../data/processed/sales_clean.csv')

In [ ]:
#load into SQL lite database
conn = sqlite3.connect('../data/sales.db')
df.to_sql('orders', conn, if_exists='replace', index=False)


9792

In [4]:
print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("Database connection established")

Data loaded: 9792 rows, 21 columns
Database connection established


## Monthly Revenue Trend


In [5]:
monthly_revenue = pd.read_sql("""
    SELECT 
        order_month,
        ROUND(SUM(sales), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS num_orders
    FROM orders
    GROUP BY order_month
    ORDER BY order_month
""", conn)

print(monthly_revenue.head(10))

  order_month  total_revenue  num_orders
0     2015-01       14205.71          30
1     2015-02        4519.89          28
2     2015-03       55205.80          69
3     2015-04       27625.48          63
4     2015-05       23644.30          68
5     2015-06       34322.94          64
6     2015-07       33781.54          64
7     2015-08       27117.54          70
8     2015-09       81623.53         129
9     2015-10       31453.39          78


## Revenue by category

In [6]:
category_revenue = pd.read_sql("""
    SELECT 
        category,
        ROUND(SUM(sales), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS num_orders,
        ROUND(AVG(sales), 2) AS avg_order_value
    FROM orders
    GROUP BY category
    ORDER BY total_revenue DESC
""", conn)

print(category_revenue)

          category  total_revenue  num_orders  avg_order_value
0       Technology      827061.58        1519           456.69
1        Furniture      728090.82        1727           350.72
2  Office Supplies      704693.09        3676           119.34


What the data is telling us
Category revenue:
- Technology has the highest average order value at $456 per order
- Office Supplies has the most orders (3,676) but lowest average at $119 — people buy many small items
- This means Technology drives revenue through high value, Office Supplies through volume

## Regional Performance

In [10]:
regional_performance = pd.read_sql("""
    SELECT 
        region,
        ROUND(SUM(sales), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS num_orders,
        ROUND(AVG(days_to_ship), 1) AS avg_days_to_ship
    FROM orders
    GROUP BY region
    ORDER BY total_revenue DESC
""", conn)

print(regional_performance)

    region  total_revenue  num_orders  avg_days_to_ship
0     West      710117.35        1587               3.9
1     East      668643.85        1369               3.9
2  Central      492646.91        1156               4.1
3    South      388437.38         810               4.0


Regional performance:

- West leads in revenue at $710k despite not having the most orders
- All regions ship in roughly 4 days — consistent operations

 ## Top 10 Customers by revenue

In [11]:
top_customers = pd.read_sql("""
    SELECT 
        customer_name,
        segment,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(sales), 2) AS lifetime_value,
        ROUND(AVG(sales), 2) AS avg_order_value
    FROM orders
    GROUP BY customer_name, segment
    ORDER BY lifetime_value DESC
    LIMIT 10
""", conn)

print(top_customers)

        customer_name      segment  total_orders  lifetime_value  \
0         Sean Miller  Home Office             5        25043.05   
1        Tamara Chand    Corporate             5        19052.22   
2        Raymond Buch     Consumer             6        15117.34   
3        Tom Ashbrook  Home Office             4        14595.62   
4       Adrian Barton     Consumer            10        14473.57   
5        Ken Lonsdale     Consumer            12        14175.23   
6        Sanjit Chand     Consumer             9        14142.33   
7        Hunter Lopez     Consumer             6        12873.30   
8        Sanjit Engle     Consumer            11        12209.44   
9  Christopher Conant     Consumer             5        12129.07   

   avg_order_value  
0          1669.54  
1          1587.68  
2           839.85  
3          1459.56  
4           723.68  
5           488.80  
6           642.83  
7          1170.30  
8           642.60  
9          1102.64  


Top customers:

- Sean Miller spent $25,043 across only 5 orders — extremely high value customer
- Ken Lonsdale made 12 orders but lower average — loyal but budget conscious

# Sales by ship mode

In [12]:
ship_mode_analysis = pd.read_sql("""
    SELECT 
        ship_mode,
        COUNT(DISTINCT order_id) AS num_orders,
        ROUND(AVG(days_to_ship), 1) AS avg_days_to_ship,
        ROUND(SUM(sales), 2) AS total_revenue
    FROM orders
    GROUP BY ship_mode
    ORDER BY avg_days_to_ship
""", conn)

print(ship_mode_analysis)

        ship_mode  num_orders  avg_days_to_ship  total_revenue
0        Same Day         261               0.0      125219.04
1     First Class         772               2.2      345523.14
2    Second Class         944               3.3      448981.08
3  Standard Class        2945               5.0     1340122.24


Ship mode:

- Standard Class handles 2,945 orders — the workhorse of the business
- Same Day averages 0 days to ship — makes sense
- Most customers tolerate 5 day shipping for the cheaper Standard Class option